In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'not run'}
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import ZeroPadding2D,Convolution2D,MaxPooling2D
from tensorflow.keras.layers import Dense,Dropout,Softmax,Flatten,Activation,BatchNormalization
from tensorflow.keras.preprocessing.image import load_img,img_to_array
from tensorflow.keras.applications.imagenet_utils import preprocess_input
import tensorflow.keras.backend as K

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'not run'}
# Define VGG_FACE_MODEL architecture
model = Sequential()
model.add(ZeroPadding2D((1,1),input_shape=(224,224, 3)))
model.add(Convolution2D(64, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
model.add(ZeroPadding2D((1,1)))	
model.add(Convolution2D(128, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(256, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(ZeroPadding2D((1,1)))
model.add(Convolution2D(512, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2), strides=(2,2)))
model.add(Convolution2D(4096, (7, 7), activation='relu'))
model.add(Dropout(0.5))
model.add(Convolution2D(4096, (1, 1), activation='relu'))
model.add(Dropout(0.5))
model.add(Convolution2D(2622, (1, 1)))
model.add(Flatten())
model.add(Activation('softmax'))

# Load VGG Face model weights
model.load_weights('data/vgg_face_weights.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/zero_padding2d.py:72: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'not run'}
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
# --- [CELL 3]: ---
# cell_state: edited
# execution_status: {'status': 'not run'}
# === BEFORE (original) ===
# # Set the main data directory where subdirectories represent classes/labels
# main_data_directory = 'data/train-data-imgs'
# 
# # Define the input size for the VGG16 model
# input_size = (224, 224)
# 
# # Create a data generator for training data
# train_datagen = ImageDataGenerator(
#     rescale=1.0/255,
#     rotation_range=20,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True,
#     zoom_range=0.2
# )
# 
# train_generator = train_datagen.flow_from_directory(
#     main_data_directory,
#     target_size=input_size,
#     batch_size=32,
#     class_mode='categorical',
#     shuffle=True
# )
# 
# # Load the VGG16 model without the top classification layer
# base_model = VGG16(weights='imagenet', include_top=False,classes=7)
# 
# # Make the layers in the base model non-trainable
# for layer in base_model.layers:
#     layer.trainable = False
# 
# # Compile the model with an appropriate optimizer, loss function, and metrics
# base_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# 
# # Load the previously saved model weights
# base_model.load_weights('data/vgg_face_weights.h5')
# 
# # Continue training the model
# base_model.fit(
#     train_generator,
#     steps_per_epoch=len(train_generator),
#     epochs=10,  # You can adjust the number of epochs
# )
# 
# # Save the model after additional training
# base_model.save('data/updated_vgg_face_weights.h5')

# === AFTER (edited) ===
main_data_directory = 'data/train-data-imgs'


input_size = (224, 224)


train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

train_generator = train_datagen.flow_from_directory(
    main_data_directory,
    target_size=input_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)


base_model = VGG16(weights='imagenet', include_top=False)


for layer in base_model.layers:
    layer.trainable = False


from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Add custom classification layers on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
predictions = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=10,
)


model.save('data/updated_vgg_face_weights')

Found 1387 images belonging to 7 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


44/44 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.1996 - loss: 1.9134
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 167us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 3/10


/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


44/44 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.3080 - loss: 1.6862
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 69us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.3717 - loss: 1.5925
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 61us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 7/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.4387 - loss: 1.5311
Epoch 8/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 67us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 9/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.4491 - loss: 1.4723
Epoch 10/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 65us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00


ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=data/updated_vgg_face_weights.

In [5]:
# Verify the fix built a compatible 7-class classifier head on top of VGG16.
assert base_model is not None, "Expected base_model to be defined by the previous cell"
assert base_model.output_shape[-1] == 7, (
    f"Expected 7 output classes, got output shape {base_model.output_shape}"
)

last_layer = base_model.layers[-1]
assert getattr(last_layer, "units", None) == 7, "Final layer must be a 7-unit classifier"
activation = getattr(last_layer, "activation", None)
assert activation is not None and activation.__name__ == "softmax", (
    "Final layer activation must be softmax for categorical classification"
)

AssertionError: Expected 7 output classes, got output shape (None, None, None, 512)